# 2013 vs 2100 最適作物比較（変更した国の一覧付き）

`optimal_crop_maps_rice088.ipynb` の「2013との比較」と同じ処理を**絶対参照**で実行。

- **絶対パス**: 下記の `REGRESSION_EXCEL` と `PREDICT_CSV` を必要に応じて変更。
- 供給係数: コーン 0.85, 小麦 0.80, 米 0.8

In [2]:
import pandas as pd
import numpy as np
try:
    import plotly.express as px
    HAS_PLOTLY = True
except ModuleNotFoundError:
    px = None
    HAS_PLOTLY = False
    print("地図表示には plotly が必要です。ターミナルで: pip install plotly")

# ========== 絶対参照（必要に応じてパスを変更） ==========
REGRESSION_EXCEL = r"C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット\regression_df_20260112_233447.xlsx"
PREDICT_CSV = r"C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット\predict_2100_bootstrap_results.csv"

ITEMS = ['Maize (corn)', 'Rice', 'Wheat']
KCAL_PER_KG = {'Maize (corn)': 3960, 'Rice': 3840, 'Wheat': 3750}
SUPPLY_COEF = {'Maize (corn)': 0.85, 'Rice': 0.8, 'Wheat': 0.80}

COUNTRY_65 = {
    'Greece', 'Zambia', 'Italy', 'Thailand', 'Canada', 'Peru', 'United States of America',
    'Dominican Republic', 'Republic of Korea', 'Kazakhstan', 'El Salvador', 'Suriname', 'Eswatini',
    'Oman', 'Austria', 'Kuwait', 'Costa Rica', 'Argentina', 'Namibia', 'Chile', 'Spain', 'Panama',
    'Ecuador', 'Myanmar', 'Belgium', 'Qatar', 'Uruguay', 'Guyana', 'Morocco', 'Honduras', 'Australia',
    'Mauritania', 'Colombia', 'France', 'Japan', 'Haiti', 'New Zealand', 'Fiji', 'Niger', 'Switzerland',
    'Poland', 'Russian Federation', 'Somalia', 'Guatemala', 'Comoros', 'Mexico', 'Portugal', 'Armenia',
    'Türkiye', 'Slovenia', 'China', 'Syrian Arab Republic', 'Trinidad and Tobago', 'Ghana', 'North Macedonia',
    'Belarus', 'Togo', 'Rwanda', 'Libya', 'Guinea-Bissau', 'Burundi', 'Israel', 'Germany', 'Bangladesh', 'Nicaragua',
}

NAME_MAP = {
    'United States of America': 'United States', 'Republic of Korea': 'South Korea',
    'Russian Federation': 'Russia', 'Syrian Arab Republic': 'Syria', 'Türkiye': 'Turkey',
    'Viet Nam': 'Vietnam', "Côte d'Ivoire": 'Ivory Coast',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
}

ISO3_MAP = {
    'Greece': 'GRC', 'Zambia': 'ZMB', 'Italy': 'ITA', 'Thailand': 'THA', 'Canada': 'CAN', 'Peru': 'PER',
    'United States of America': 'USA', 'Dominican Republic': 'DOM', 'Republic of Korea': 'KOR', 'Kazakhstan': 'KAZ',
    'El Salvador': 'SLV', 'Suriname': 'SUR', 'Eswatini': 'SWZ', 'Oman': 'OMN', 'Austria': 'AUT', 'Kuwait': 'KWT',
    'Costa Rica': 'CRI', 'Argentina': 'ARG', 'Namibia': 'NAM', 'Chile': 'CHL', 'Spain': 'ESP', 'Panama': 'PAN',
    'Ecuador': 'ECU', 'Myanmar': 'MMR', 'Belgium': 'BEL', 'Qatar': 'QAT', 'Uruguay': 'URY', 'Guyana': 'GUY',
    'Morocco': 'MAR', 'Honduras': 'HND', 'Australia': 'AUS', 'Mauritania': 'MRT', 'Colombia': 'COL', 'France': 'FRA',
    'Japan': 'JPN', 'Haiti': 'HTI', 'New Zealand': 'NZL', 'Fiji': 'FJI', 'Niger': 'NER', 'Switzerland': 'CHE',
    'Poland': 'POL', 'Russian Federation': 'RUS', 'Somalia': 'SOM', 'Guatemala': 'GTM', 'Comoros': 'COM',
    'Mexico': 'MEX', 'Portugal': 'PRT', 'Armenia': 'ARM', 'Türkiye': 'TUR', 'Slovenia': 'SVN', 'China': 'CHN',
    'Syrian Arab Republic': 'SYR', 'Trinidad and Tobago': 'TTO', 'Ghana': 'GHA', 'North Macedonia': 'MKD',
    'Belarus': 'BLR', 'Togo': 'TGO', 'Rwanda': 'RWA', 'Libya': 'LBY', 'Guinea-Bissau': 'GNB', 'Burundi': 'BDI',
    'Israel': 'ISR', 'Germany': 'DEU', 'Bangladesh': 'BGD', 'Nicaragua': 'NIC',
}

def yield_to_kcal(df, yield_col):
    out = df.copy()
    out['kcal_per_ha'] = out[yield_col] * out['Item'].map(KCAL_PER_KG)
    out['effective_kcal'] = out['kcal_per_ha'] * out['Item'].map(SUPPLY_COEF)
    return out

def optimal_crop_per_country(df):
    idx = df.groupby('Area')['effective_kcal'].idxmax()
    best = df.loc[idx, ['Area', 'Item', 'effective_kcal', 'kcal_per_ha']].copy()
    best = best.rename(columns={'Item': 'optimal_crop'})
    return best[['Area', 'optimal_crop', 'effective_kcal', 'kcal_per_ha']]

In [3]:
# 絶対パスで読み込み
reg = pd.read_excel(REGRESSION_EXCEL)
pred = pd.read_csv(PREDICT_CSV)

print('供給係数: Maize 0.85, Wheat 0.80, Rice 0.8')
print('regression_df:', reg.shape, '| predict:', pred.shape)

# 2013
y13 = reg[(reg['Year'].between(2011, 2013)) & (reg['Item'].isin(ITEMS))][['Area', 'Item', 'Yield']].copy()
y13 = y13[y13['Area'].isin(COUNTRY_65)]
y13 = y13.groupby(['Area', 'Item'], as_index=False)['Yield'].mean()
y13 = y13.rename(columns={'Yield': 'yield_kg'})
y13 = yield_to_kcal(y13, 'yield_kg')
opt_2013 = optimal_crop_per_country(y13)
opt_2013['scenario'] = '2013 (2011–2013平均)'

# 2100 各シナリオ
p = pred[pred['Area'].isin(COUNTRY_65)].copy()
p370m = p[p['scenario'] == 'SSP370'][['Area', 'Item', 'mean_yield']].rename(columns={'mean_yield': 'y'})
p370m = yield_to_kcal(p370m, 'y')
opt_370m = optimal_crop_per_country(p370m)
opt_370m['scenario'] = '2100 SSP370 平均'

p370c = p[p['scenario'] == 'SSP370'][['Area', 'Item', 'ci95_lower']].rename(columns={'ci95_lower': 'y'})
p370c = yield_to_kcal(p370c, 'y')
opt_370c = optimal_crop_per_country(p370c)
opt_370c['scenario'] = '2100 SSP370 95%CI下限'

p245m = p[p['scenario'] == 'SSP245'][['Area', 'Item', 'mean_yield']].rename(columns={'mean_yield': 'y'})
p245m = yield_to_kcal(p245m, 'y')
opt_245m = optimal_crop_per_country(p245m)
opt_245m['scenario'] = '2100 SSP245 平均'

p245c = p[p['scenario'] == 'SSP245'][['Area', 'Item', 'ci95_lower']].rename(columns={'ci95_lower': 'y'})
p245c = yield_to_kcal(p245c, 'y')
opt_245c = optimal_crop_per_country(p245c)
opt_245c['scenario'] = '2100 SSP245 95%CI下限'

all_opt = pd.concat([opt_2013, opt_370m, opt_370c, opt_245m, opt_245c], ignore_index=True)
all_opt['Country'] = all_opt['Area'].replace(NAME_MAP)
all_opt['ISO3'] = all_opt['Area'].map(ISO3_MAP)
order = ['2013 (2011–2013平均)', '2100 SSP370 平均', '2100 SSP370 95%CI下限', '2100 SSP245 平均', '2100 SSP245 95%CI下限']
all_opt['scenario'] = pd.Categorical(all_opt['scenario'], categories=order, ordered=True)

供給係数: Maize 0.85, Wheat 0.80, Rice 0.8
regression_df: (16388, 8) | predict: (936, 5)


In [4]:
base = opt_2013[['Area', 'optimal_crop']].rename(columns={'optimal_crop': 'opt_2013'})
future_scenarios = ['2100 SSP370 平均', '2100 SSP370 95%CI下限', '2100 SSP245 平均', '2100 SSP245 95%CI下限']

print('=== 2013との比較 ===\n')
for s in future_scenarios:
    fut = all_opt[all_opt['scenario'] == s][['Area', 'optimal_crop']].copy()
    fut = fut.rename(columns={'optimal_crop': 'opt_2100'})
    m = base.merge(fut, on='Area')
    same = (m['opt_2013'] == m['opt_2100']).sum()
    chg = len(m) - same
    print(f'{s}:')
    print(f'  2013と同じ最適作物: {same}か国')
    print(f'  2013から変更: {chg}か国')
    chgd = m[m['opt_2013'] != m['opt_2100']]
    if len(chgd) > 0:
        trans = chgd.groupby(['opt_2013', 'opt_2100']).size().reset_index(name='n')
        print('  変更内訳（2013 → 2100）:')
        for _, r in trans.iterrows():
            print(f'    {r["opt_2013"]} → {r["opt_2100"]}: {int(r["n"])}か国')
            countries = chgd[(chgd['opt_2013'] == r['opt_2013']) & (chgd['opt_2100'] == r['opt_2100'])]['Area'].sort_values().tolist()
            print(f'      → {", ".join(countries)}')
        print('  変更した国（一覧）: ' + ', '.join(chgd['Area'].sort_values().tolist()))
    print()

=== 2013との比較 ===

2100 SSP370 平均:
  2013と同じ最適作物: 58か国
  2013から変更: 7か国
  変更内訳（2013 → 2100）:
    Maize (corn) → Rice: 3か国
      → Myanmar, Syrian Arab Republic, Türkiye
    Rice → Maize (corn): 2か国
      → North Macedonia, Russian Federation
    Rice → Wheat: 1か国
      → Mexico
    Wheat → Rice: 1か国
      → Namibia
  変更した国（一覧）: Mexico, Myanmar, Namibia, North Macedonia, Russian Federation, Syrian Arab Republic, Türkiye

2100 SSP370 95%CI下限:
  2013と同じ最適作物: 59か国
  2013から変更: 6か国
  変更内訳（2013 → 2100）:
    Maize (corn) → Rice: 3か国
      → Myanmar, Syrian Arab Republic, Türkiye
    Rice → Maize (corn): 1か国
      → Russian Federation
    Rice → Wheat: 1か国
      → Mexico
    Wheat → Rice: 1か国
      → Namibia
  変更した国（一覧）: Mexico, Myanmar, Namibia, Russian Federation, Syrian Arab Republic, Türkiye

2100 SSP245 平均:
  2013と同じ最適作物: 59か国
  2013から変更: 6か国
  変更内訳（2013 → 2100）:
    Maize (corn) → Rice: 2か国
      → Myanmar, Syrian Arab Republic
    Rice → Maize (corn): 2か国
      → North Macedonia, Russian F

### 最適作物マップ（5シナリオ）

各シナリオで国別の最適作物（Maize / Rice / Wheat）をチョロプレスで表示。赤＝コーン、緑＝米、青＝小麦。

In [1]:
if not HAS_PLOTLY:
    print("地図を表示するには: pip install plotly を実行してからカーネルを再起動してください。")
else:
    color_map = {'Maize (corn)': '#e74c3c', 'Rice': '#27ae60', 'Wheat': '#3498db'}
    for s in order:
        d = all_opt[all_opt['scenario'] == s].dropna(subset=['ISO3'])
        fig = px.choropleth(
            d,
            locations='ISO3',
            locationmode='ISO-3',
            color='optimal_crop',
            color_discrete_map=color_map,
            hover_name='Area',
            hover_data={'optimal_crop': True, 'effective_kcal': ':,.0f', 'kcal_per_ha': ':,.0f', 'ISO3': False},
            title=f'最適作物（供給係数 コーン0.85, 小麦0.8, 米0.8） — {s}  ({len(d)}か国)'
        )
        fig.update_layout(
            margin=dict(l=0, r=0, t=50, b=0),
            height=420,
            geo=dict(showframe=False, showcoastlines=True, landcolor='#95a5a6'),
            legend_title='最適作物'
        )
        fig.show()
    print('5 maps (最適作物) done.')

NameError: name 'HAS_PLOTLY' is not defined

### 2013 vs 2100 最適作物カロリー変化（パーセント・グラデーション）

各2100シナリオについて、2013の最適作物の effective_kcal との**パーセント変化**を地図表示。青＝減少、白＝0、オレンジ＝増加。±40%でクリップ。

In [ ]:
if not HAS_PLOTLY:
    print("地図を表示するには: pip install plotly を実行してからカーネルを再起動してください。")
else:
    base_kcal = opt_2013[['Area', 'effective_kcal']].rename(columns={'effective_kcal': 'kcal_2013'})
    scale_div = [[0, '#3498db'], [0.5, '#ecf0f1'], [1, '#e67e22']]
    for s in future_scenarios:
        fut = all_opt[all_opt['scenario'] == s][['Area', 'effective_kcal']].rename(columns={'effective_kcal': 'kcal_2100'})
        m = base_kcal.merge(fut, on='Area')
        m['diff'] = m['kcal_2100'] - m['kcal_2013']
        m['pct_change'] = np.where(m['kcal_2013'] > 0, m['diff'] / m['kcal_2013'] * 100, np.nan)
        m['ISO3'] = m['Area'].map(ISO3_MAP)
        d = m.dropna(subset=['ISO3', 'pct_change'])
        fig = px.choropleth(
            d,
            locations='ISO3',
            locationmode='ISO-3',
            color='pct_change',
            color_continuous_scale=scale_div,
            color_continuous_midpoint=0,
            range_color=[-40, 40],
            hover_name='Area',
            hover_data={'pct_change': ':.1f', 'kcal_2013': ':,.0f', 'kcal_2100': ':,.0f', 'diff': ':,.0f', 'ISO3': False},
            title=f'2013 vs {s} 最適作物カロリー変化（%）'
        )
        fig.update_layout(
            margin=dict(l=0, r=0, t=50, b=0),
            height=420,
            geo=dict(showframe=False, showcoastlines=True, landcolor='#95a5a6'),
            coloraxis_colorbar_title='2013比（%）'
        )
        fig.show()
    print('4 maps (2013 vs 2100 パーセント変化) done.')

NameError: name 'HAS_PLOTLY' is not defined

### 最適 vs 2番目 の差（%）

国ごとに最適作物と2番目に高い作物の effective_kcal の差を % で表示。0–20% グラデーション、20%超は濃色。

In [ ]:
if not HAS_PLOTLY:
    print("地図を表示するには: pip install plotly を実行してからカーネルを再起動してください。")
else:
    def best_vs_second(df):
        """Area, Item, effective_kcal の df → 最適・2番目・pct_diff"""
        rows = []
        for area, g in df.groupby('Area'):
            g = g.sort_values('effective_kcal', ascending=False)
            if len(g) < 2:
                continue
            r1, r2 = g.iloc[0], g.iloc[1]
            best_k, second_k = r1['effective_kcal'], r2['effective_kcal']
            if second_k <= 0:
                continue
            pct = (best_k - second_k) / second_k * 100
            rows.append({
                'Area': area,
                'optimal_crop': r1['Item'],
                'second_crop': r2['Item'],
                'best_kcal': best_k,
                'second_kcal': second_k,
                'pct_diff': pct,
            })
        return pd.DataFrame(rows)

    g2013 = best_vs_second(y13)
    g2013['scenario'] = '2013 (2011–2013平均)'
    gap_all = pd.concat([
        g2013,
        best_vs_second(p370m).assign(scenario='2100 SSP370 平均'),
        best_vs_second(p370c).assign(scenario='2100 SSP370 95%CI下限'),
        best_vs_second(p245m).assign(scenario='2100 SSP245 平均'),
        best_vs_second(p245c).assign(scenario='2100 SSP245 95%CI下限'),
    ], ignore_index=True)
    gap_all['ISO3'] = gap_all['Area'].map(ISO3_MAP)
    scale_gap = [[0, '#ecf0f1'], [1, '#2c3e50']]
    for s in order:
        d = gap_all[gap_all['scenario'] == s].dropna(subset=['ISO3'])
        fig = px.choropleth(
            d,
            locations='ISO3',
            locationmode='ISO-3',
            color='pct_diff',
            range_color=[0, 20],
            color_continuous_scale=scale_gap,
            hover_name='Area',
            hover_data={'optimal_crop': True, 'second_crop': True, 'pct_diff': ':.1f', 'best_kcal': ':,.0f', 'second_kcal': ':,.0f', 'ISO3': False},
            title=f'最適 vs 2番目 の差（%） — {s}  [0–20% グラデーション、20%超は濃色]'
        )
        fig.update_layout(
            margin=dict(l=0, r=0, t=50, b=0),
            height=420,
            geo=dict(showframe=False, showcoastlines=True, landcolor='#95a5a6'),
            coloraxis_colorbar_title='差（%）'
        )
        fig.show()
    print('5 maps (最適 vs 2番目 の差 %) done.')